# Notebook 02: Tier-1 Training (LightGBM + IsolationForest)

## Objective
Binary classification: **Benign vs Anomaly**

Target: **Recall > 95%** on anomaly class

### Models
- **Primary**: LightGBM
- **Secondary**: IsolationForest

### Decision Logic
```
if (LightGBM_probability > 0.30 OR IsolationForest detects anomaly):
    status = "anomaly"
else:
    status = "benign"
```

### Training Requirements
- Class balancing
- Save trained models
- Save preprocessing pipeline
- Save feature order
- Save thresholds
- Evaluation metrics: Precision, Recall, F1, Confusion Matrix, ROC-AUC

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import joblib
import logging
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_score, recall_score, f1_score
)
from sklearn.model_selection import cross_val_score
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

with open('config/config.json', 'r') as f:
    config = json.load(f)

PROCESSED_DIR = os.path.join(config['dataset']['data_dir'], 'processed')
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

## Step 1: Load Processed Data

In [ ]:
X_train = pd.read_csv(os.path.join(PROCESSED_DIR, 'tier1_X_train.csv'))
X_test = pd.read_csv(os.path.join(PROCESSED_DIR, 'tier1_X_test.csv'))
y_train = pd.read_csv(os.path.join(PROCESSED_DIR, 'tier1_y_train.csv')).values.ravel()
y_test = pd.read_csv(os.path.join(PROCESSED_DIR, 'tier1_y_test.csv')).values.ravel()
feature_order = joblib.load(os.path.join(PROCESSED_DIR, 'feature_order.joblib'))

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Label distribution (train):')
print(f'  Benign (0): {(y_train == 0).sum()}')
print(f'  Anomaly (1): {(y_train == 1).sum()}')
print(f'Feature count: {len(feature_order)}')

## Step 2: Preprocessing & Class Balancing

In [ ]:
# Fill remaining NaN/Inf values
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# Ensure feature order
X_train = X_train[feature_order]
X_test = X_test[feature_order]

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Class weight computation for LightGBM
benign_count = (y_train == 0).sum()
anomaly_count = (y_train == 1).sum()
scale_pos_weight = benign_count / anomaly_count
print(f'Class ratio (benign/anomaly): {scale_pos_weight:.2f}')
print(f'Benign: {benign_count}, Anomaly: {anomaly_count}')

# Save preprocessor
joblib.dump(scaler, os.path.join(MODELS_DIR, 'tier1_preprocessor.joblib'))
print('Preprocessor saved.')

## Step 3: Train LightGBM

In [ ]:
lgbm_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'max_depth': -1,
    'learning_rate': 0.05,
    'n_estimators': 500,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'verbose': -1,
    'random_state': 42
}

lgbm_model = lgb.LGBMClassifier(**lgbm_params)
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)

y_pred_lgbm = lgbm_model.predict(X_test)
y_proba_lgbm = lgbm_model.predict_proba(X_test)[:, 1]

print('\nLightGBM Results:')
print(classification_report(y_test, y_pred_lgbm, target_names=['Benign', 'Anomaly']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_lgbm):.4f}')

## Step 4: Train Isolation Forest

In [ ]:
iforest = IsolationForest(
    n_estimators=200,
    contamination=float(anomaly_count) / float(benign_count + anomaly_count),
    random_state=42,
    n_jobs=-1
)

iforest.fit(X_train_scaled)

# IsolationForest: -1 = anomaly, 1 = normal
y_pred_if = iforest.predict(X_test_scaled)
y_pred_if_binary = (y_pred_if == -1).astype(int)

print('IsolationForest Results:')
print(classification_report(y_test, y_pred_if_binary, target_names=['Benign', 'Anomaly']))

## Step 5: Combined Decision Logic Evaluation

```
if (LightGBM_probability > 0.30 OR IsolationForest detects anomaly):
    status = "anomaly"
else:
    status = "benign"
```

In [ ]:
LGBM_THRESHOLD = config['tier1']['lgbm_threshold']  # 0.30

y_combined = np.zeros(len(y_test), dtype=int)

lgbm_anomaly = y_proba_lgbm > LGBM_THRESHOLD
if_anomaly = y_pred_if == -1

y_combined[lgbm_anomaly | if_anomaly] = 1

print(f'Combined Decision Logic (LGBM threshold={LGBM_THRESHOLD}):')
print(f'\nLightGBM anomaly detections: {lgbm_anomaly.sum()}')
print(f'IsolationForest anomaly detections: {if_anomaly.sum()}')
print(f'Combined anomaly detections: {y_combined.sum()}')
print(f'Actual anomalies: {y_test.sum()}')

print(f'\nCombined Model Results:')
print(classification_report(y_test, y_combined, target_names=['Benign', 'Anomaly']))

recall = recall_score(y_test, y_combined)
precision = precision_score(y_test, y_combined)
f1 = f1_score(y_test, y_combined)

print(f'Recall: {recall:.4f} (target > 0.95)')
print(f'Precision: {precision:.4f}')
print(f'F1: {f1:.4f}')

# Confusion Matrix
cm = confusion_matrix(y_test, y_combined)
print(f'\nConfusion Matrix:')
print(f'  TP: {cm[1][1]}, FP: {cm[0][1]}')
print(f'  FN: {cm[1][0]}, TN: {cm[0][0]}')

## Step 6: Threshold Tuning (if recall < 95%)

In [ ]:
import json

if recall < 0.95:
    print('Recall below 95% target. Tuning threshold...')
    best_threshold = LGBM_THRESHOLD
    best_recall = recall
    
    for thresh in np.arange(0.05, 0.50, 0.01):
        lgbm_anom = y_proba_lgbm > thresh
        combined = np.zeros(len(y_test), dtype=int)
        combined[lgbm_anom | if_anomaly] = 1
        r = recall_score(y_test, combined)
        p = precision_score(y_test, combined)
        if r >= 0.95 and p > precision:
            best_threshold = thresh
            best_recall = r
            print(f'  threshold={thresh:.2f}: recall={r:.4f}, precision={p:.4f}')
    
    LGBM_THRESHOLD = best_threshold
    print(f'\nBest threshold: {LGBM_THRESHOLD}')
else:
    print(f'Recall {recall:.4f} >= 0.95 target. Current threshold sufficient.')

# Final combined prediction with tuned threshold
y_final = np.zeros(len(y_test), dtype=int)
y_final[(y_proba_lgbm > LGBM_THRESHOLD) | if_anomaly] = 1
print(f'\nFinal Results with threshold {LGBM_THRESHOLD}:')
print(classification_report(y_test, y_final, target_names=['Benign', 'Anomaly']))
print(f'Final Recall: {recall_score(y_test, y_final):.4f}')

## Step 7: SHAP Explanation

In [ ]:
explainer = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_test[:100])

# Summary plot
shap.summary_plot(shap_values, X_test[:100], plot_type='bar')
plt.title('Tier-1 LightGBM Feature Importance (SHAP)')
plt.tight_layout()
plt.savefig('models/tier1_shap_summary.png', dpi=150)
plt.show()

joblib.dump(explainer, os.path.join(MODELS_DIR, 'tier1_shap_explainer.joblib'))
print('SHAP explainer saved.')

## Step 8: Save Models & Artifacts

In [ ]:
joblib.dump(lgbm_model, os.path.join(MODELS_DIR, 'tier1_lgbm.joblib'))
joblib.dump(iforest, os.path.join(MODELS_DIR, 'tier1_iforest.joblib'))
joblib.dump(scaler, os.path.join(MODELS_DIR, 'tier1_preprocessor.joblib'))
joblib.dump(feature_order, os.path.join(MODELS_DIR, 'tier1_feature_order.joblib'))

# Save thresholds
thresholds = {
    'lgbm_threshold': LGBM_THRESHOLD,
    'target_recall': 0.95
}
joblib.dump(thresholds, os.path.join(MODELS_DIR, 'tier1_thresholds.joblib'))

# Update config with tuned threshold
config['tier1']['lgbm_threshold'] = LGBM_THRESHOLD
with open('config/config.json', 'w') as f:
    json.dump(config, f, indent=4)

print('Tier-1 models and artifacts saved:')
print(f'  tier1_lgbm.joblib')
print(f'  tier1_iforest.joblib')
print(f'  tier1_preprocessor.joblib')
print(f'  tier1_feature_order.joblib')
print(f'  tier1_thresholds.joblib')
print(f'  tier1_shap_explainer.joblib')
print(f'  LGBM threshold: {LGBM_THRESHOLD}')